In [7]:
from langgraph.graph import StateGraph , START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()

In [ ]:
llm = ChatOpenAI()

In [6]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state:ChatState):
    messages=state['messages']
    
    #send to llm 
    response = llm.invoke(messages)
    #response store state
    return {'messages': [response]}



In [ ]:
checkpoint= MemorySaver()
graph= StateGraph(ChatState)

#add nodes
graph.add_node('chat_node', chat_node)

#add edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot=graph.compile(checkpointer = checkpoint)


In [ ]:
initial_state={
    'messages': [HumanMessage(content='what is object oriented programming')]
}

In [ ]:
chatbot.invoke(initial_state)['messages'][-1].content

In [ ]:
thread_id="1"
while True:
    user_message= input("Type here:   ")
    print('User', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
    config={'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content= user_message)]}, config=config) 
